# 04-1 실습 — 경로와 안전한 작업 범위

경로 문자열을 곧바로 열지 않고 기준 위치, 대상 종류, 허용 범위를 차례로 확인합니다. 모든 파일은 실행할 때마다 새로 만들어지는 임시 디렉터리 안에만 생성됩니다.

## Goal

- Path 객체와 실제 파일을 구분합니다.
- 현재 작업 디렉터리를 기준으로 상대 경로와 절대 경로를 비교합니다.
- 경로 구성요소, resolve, 대상 종류, 정렬된 탐색과 메타데이터를 확인합니다.
- 사용자 입력을 임시 실습 디렉터리 안으로 제한하는 resolve_under를 완성해 사용합니다.
- 정상·오류·경계 입력을 assert로 스스로 점검합니다.

## Setup

Python 3.9 이상과 표준 라이브러리만 사용합니다. LAB_ROOT는 TemporaryDirectory가 관리하므로 개인 문서나 프로젝트 파일을 건드리지 않습니다.

In [ ]:
import sys
from datetime import datetime, timezone
from pathlib import Path
from tempfile import TemporaryDirectory

assert sys.version_info >= (3, 9), 'Python 3.9 이상이 필요합니다.'

_lab_context = TemporaryDirectory(prefix='chapter04-paths-')
LAB_ROOT = Path(_lab_context.name).resolve(strict=True)
INPUT_DIR = LAB_ROOT / 'input'
OUTPUT_DIR = LAB_ROOT / 'output'
NESTED_DIR = INPUT_DIR / 'nested'
EMPTY_DIR = LAB_ROOT / 'empty'

for directory in (INPUT_DIR, OUTPUT_DIR, NESTED_DIR, EMPTY_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def expect_raises(expected_exception, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except expected_exception as exc:
        return exc
    except Exception as exc:
        raise AssertionError(
            f'{expected_exception.__name__} 대신 {type(exc).__name__} 발생'
        ) from exc
    raise AssertionError(f'{expected_exception.__name__}이 발생하지 않았습니다.')

print('격리된 실습 루트:', LAB_ROOT)

## Steps

### 1. Path 객체와 기준 위치 확인

Path를 만드는 것만으로 파일이 생기지는 않습니다. 상대 경로는 현재 작업 디렉터리를 기준으로 해석되지만, 이 실습의 실제 입출력은 명시적인 LAB_ROOT만 사용합니다.

In [ ]:
planned_path = Path('data') / 'users.txt'
planned_output = OUTPUT_DIR / 'report.json'

assert not planned_output.exists()
assert not planned_path.is_absolute()
assert LAB_ROOT.is_absolute()

print('현재 작업 디렉터리:', Path.cwd())
print('상대 경로 표현:', planned_path)
print('명시적인 실습 경로:', planned_output)

### 2. 재현 가능한 파일 구조 만들기

일반 파일, 디렉터리, 빈 디렉터리를 만들고 가능한 환경에서는 심볼릭 링크도 만듭니다. 링크 생성 권한이 없는 환경에서는 링크 검사만 건너뜁니다.

In [ ]:
(INPUT_DIR / 'alpha.txt').write_text('alpha\n', encoding='utf-8')
(INPUT_DIR / 'table.csv').write_text('name,score\nA,10\n', encoding='utf-8')
(NESTED_DIR / 'gamma.txt').write_text('gamma\n', encoding='utf-8')
(INPUT_DIR / 'folder.txt').mkdir()

link_path = INPUT_DIR / 'alpha-link.txt'
try:
    link_path.symlink_to(INPUT_DIR / 'alpha.txt')
except (NotImplementedError, OSError):
    symlink_supported = False
else:
    symlink_supported = True
    assert link_path.is_symlink()
    assert link_path.is_file()

print('심볼릭 링크 실습 가능:', symlink_supported)

### 3. 구성요소·정리·탐색·메타데이터 확인

탐색 결과는 파일시스템의 기본 순서에 기대지 않고 정렬합니다. 수정 시각은 환경마다 정밀도가 다르므로 특정 값과 같다고 비교하지 않습니다.

In [ ]:
compound = Path('reports') / 'archive.tar.gz'
assert compound.name == 'archive.tar.gz'
assert compound.stem == 'archive.tar'
assert compound.suffix == '.gz'
assert compound.suffixes == ['.tar', '.gz']
assert compound.parent == Path('reports')

messy = INPUT_DIR / 'nested' / '..' / 'alpha.txt'
assert messy.resolve(strict=True) == (INPUT_DIR / 'alpha.txt').resolve(strict=True)
future = (OUTPUT_DIR / 'future.txt').resolve(strict=False)
assert future.parent == OUTPUT_DIR

entries = sorted(INPUT_DIR.iterdir(), key=lambda path: path.name)
assert [path.name for path in entries] == sorted(path.name for path in entries)

info = (INPUT_DIR / 'alpha.txt').stat()
modified_at = datetime.fromtimestamp(info.st_mtime, tz=timezone.utc)
assert info.st_size == len('alpha\n'.encode('utf-8'))
assert modified_at.utcoffset().total_seconds() == 0

print('항목:', [path.name for path in entries])
print('alpha.txt 크기:', info.st_size, 'bytes')
print('수정 시각(UTC):', modified_at.isoformat())

### 4. 허용된 기준 디렉터리 안으로 경로 제한

빈 입력과 절대 경로를 먼저 거부하고, 정리된 후보가 기준 디렉터리 내부인지 경로 구성요소 단위로 확인합니다. 문자열 startswith 비교는 사용하지 않습니다.

In [ ]:
def resolve_under(base, user_value, *, must_exist=False):
    if not isinstance(user_value, str):
        raise TypeError('경로 입력은 문자열이어야 합니다.')
    if not user_value.strip():
        raise ValueError('경로가 비어 있습니다.')

    raw_path = Path(user_value)
    if raw_path.is_absolute():
        raise ValueError('절대 경로는 허용하지 않습니다.')

    resolved_base = Path(base).resolve(strict=True)
    candidate = (resolved_base / raw_path).resolve(strict=must_exist)
    if not candidate.is_relative_to(resolved_base):
        raise ValueError('허용된 디렉터리 밖의 경로입니다.')
    return candidate

safe_path = resolve_under(LAB_ROOT, 'input/../input/alpha.txt', must_exist=True)
assert safe_path == (INPUT_DIR / 'alpha.txt').resolve(strict=True)
print('검증된 경로:', safe_path.relative_to(LAB_ROOT))

### 5. 안전한 텍스트 파일 목록 만들기

한 단계 아래의 .txt 일반 파일만 선택하고 디렉터리와 링크를 제외합니다. 아래 chosen_directory를 input 또는 input/nested로 바꾸어 결과 차이를 확인하세요.

In [ ]:
def collect_text_files(base, user_directory):
    root = Path(base).resolve(strict=True)
    directory = resolve_under(root, user_directory, must_exist=True)
    if not directory.is_dir():
        raise NotADirectoryError(directory)

    records = []
    for child in sorted(directory.iterdir(), key=lambda path: path.name):
        if child.is_symlink() or not child.is_file() or child.suffix != '.txt':
            continue
        records.append({
            'name': child.name,
            'relative_path': child.relative_to(root).as_posix(),
            'size_bytes': child.stat().st_size,
        })

    return sorted(records, key=lambda record: record['relative_path'])

chosen_directory = 'input'  # 'input/nested'로 바꾸어 다시 실행해 보세요.
expected_names = {
    'input': ['alpha.txt'],
    'input/nested': ['gamma.txt'],
}
assert chosen_directory in expected_names

records = collect_text_files(LAB_ROOT, chosen_directory)
assert [record['name'] for record in records] == expected_names[chosen_directory]
assert records == sorted(records, key=lambda record: record['relative_path'])
print(records)

## Checks

정상화해도 내부에 남는 경로는 허용하지만, 빈 값·절대 경로·상위 이탈·없는 입력은 서로 다른 오류로 거부합니다.

In [ ]:
assert resolve_under(LAB_ROOT, 'input/../input', must_exist=True) == INPUT_DIR
assert collect_text_files(LAB_ROOT, 'empty') == []

expect_raises(ValueError, resolve_under, LAB_ROOT, '')
expect_raises(TypeError, resolve_under, LAB_ROOT, Path('input'))
expect_raises(ValueError, resolve_under, LAB_ROOT, str(INPUT_DIR))
expect_raises(ValueError, resolve_under, LAB_ROOT, '../outside.txt')
expect_raises(FileNotFoundError, resolve_under, LAB_ROOT, 'input/missing.txt', must_exist=True)
expect_raises(NotADirectoryError, collect_text_files, LAB_ROOT, 'input/alpha.txt')

all_input_records = collect_text_files(LAB_ROOT, 'input')
assert all(record['name'] != 'folder.txt' for record in all_input_records)
if symlink_supported:
    assert all(record['name'] != 'alpha-link.txt' for record in all_input_records)

assert not planned_output.exists()
print('04-1 자기점검을 모두 통과했습니다.')

In [ ]:
_lab_context.cleanup()
assert not LAB_ROOT.exists()

## Next Steps

04-2에서는 같은 경로 검증 계약을 재사용해 파일을 생성·복사·이름 변경·삭제합니다. 운영 환경에서는 이 검사가 검사 직후 경로가 바뀌는 경쟁 조건까지 해결하지 않으며, 최소 권한과 운영체제 수준의 격리가 추가로 필요합니다.